In [13]:
import numpy as np
import pandas as pd
import easygui
from collections import defaultdict
import datetime
import tkinter as tk
from tkinter import filedialog
import os
import copy
import re
from tkinter import messagebox
import sys
from PyQt6.QtWidgets import (QApplication, QWidget, QPushButton)

In [14]:
def processing():
    global resource
    resource = defaultdict(list)
    for name in files:
        txt = pd.read_csv(name, sep = '\t',header = 2, encoding = 'ANSI', low_memory = False).drop([0])
        appendix = pd.read_csv(name, sep = '\t',encoding = 'ANSI', low_memory = False).head(0).columns[0].split(' ')[2]
        if 'kv_tm' in appendix or 'sov' in appendix or 'vrrt' in appendix:
            txt.rename(columns = {f'{txt.columns[0]}' : 'Дата Время Значение'}, inplace = True)
            txt[['Дата', 'Дата и Время','Значение']] = txt['Дата Время Значение'].str.split(expand = True).drop(columns = 3, axis = 1)
        else:
            txt.rename(columns = {f'{txt.columns[0]}':'Дата Время Значение'}, inplace = True)
            txt[['Дата', 'Дата и Время','Значение']] = txt['Дата Время Значение'].str.split(expand = True).drop(columns = [3,4], axis = 1)
        txt['Дата и Время'] = [x.split('.')[0] for x in txt['Дата и Время']]
        txt['Дата и Время'] = txt['Дата'] + ' ' + txt['Дата и Время']
        txt.drop(columns = 'Дата Время Значение', axis = 1, inplace = True)
        if 'ФОС' in appendix:
            txt['Значение'] = [0 if x == '101' else 1 for x in txt['Значение']]
            txt['Значение'] = txt['Значение'].astype('float64')
        else:
            txt['Значение'] = txt['Значение'].astype('float64')
        txt['Дата'] = pd.to_datetime(txt['Дата'], dayfirst=True).dt.date
        txt['Дата и Время'] = txt['Дата и Время'].astype('datetime64[ns]')
        txt['Дата и Время'] = pd.to_datetime(txt['Дата и Время'], format= "%d.%m.%Y %H:%M:%S")
        if 'kvop' in appendix:
            appendix = appendix.replace('kvop','ГН')
        if 'kv_tm' in appendix:
            appendix = appendix.replace('kv_tm','Вкл ТМ')
        if 'sov_' in appendix:
            appendix = appendix.replace('sov_','Время ТМ')
        resource[appendix].append(txt)

In [15]:
def GnFosSovKv():
    global count, files, count1
    count = defaultdict(list)
    count1 = defaultdict(list)
    files = easygui.fileopenbox(multiple=True)
    processing()
    for name in resource.keys():
        if 'Вкл ТМ' in name:
            count[name] = resource[name][0]['Значение'].iloc[-1] - resource[name][0]['Значение'].iloc[0]
        if 'Время ТМ' in name or 'vrrt' in name:
            count[name] = resource[name][0]['Значение'].iloc[-1] - resource[name][0]['Значение'].iloc[0]
            count[name] = f'{int(count[name]/3600)}' + f':{int((count[name]/3600 - int(count[name]/3600))*60)}' + f':{int(((count[name]/3600 - int(count[name]/3600))*60 - int((count[name]/3600 - int(count[name]/3600))*60))*60)}'
        if 'ГН' in name or 'ФОС' in name:   
            for x in range(len(resource[name][0]['Значение'].values)):
                if x < len(resource[name][0]['Значение']) - 1:
                     if resource[name][0]['Значение'].iloc[x] == 0:
                         if resource[name][0]['Значение'].iloc[x] != resource[name][0]['Значение'].iloc[x+1]:
                             count1[name].append(resource[name][0]['Значение'].iloc[x])
            count1[name] = len(count1[name])

In [16]:
GnFosSovKv()

In [17]:
def excelTM():
    global tm, tm_counts, files, proppelant
    tm = {1:{'K1':[],'K2':[]}, 2:{'K1':[],'K2':[]}, 3:{'K1':[],'K2':[]},4:{'K1':[],'K2':[]}, 5:{'K1':[],'K2':[]},
        6:{'K1':[],'K2':[]}, 7:{'K1':[],'K2':[]}, 8:{'K1':[],'K2':[]}}
    tm_counts = {'+Z':[], '-Z':[], '+Y':[],'-Y':[]}
    proppelant = {'Сумма расходов':[], 'Разность расходов':[]}
    files = easygui.fileopenbox(multiple=True)
    sheets = pd.ExcelFile(files[0])
    sheets = sheets.sheet_names
    sheets  = [x for x in sheets[:sheets.index('Наработки')]]
    for sheet in sheets:
        proppelant['Сумма расходов'].append(pd.read_excel(files[0], sheet_name = sheet).iloc[28,0])
        testing = pd.read_excel(files[0], sheet_name = sheet).drop(columns = (pd.read_excel(files[0], sheet_name = sheet).columns[0]), axis = 1)
        testing.drop(columns = (testing.columns[8::]), axis = 1, inplace = True)
        testing.rename(columns = {testing.columns[0]:1,testing.columns[1]:2,testing.columns[2]:3,testing.columns[3]:4,
                                 testing.columns[4]:5,testing.columns[5]:6,testing.columns[6]:7,testing.columns[7]:8}, inplace = True)
        testing.dropna(inplace=True)
        for i in range(len(testing.loc[testing.loc[testing[1]=="ТМ1"].index[0]+1,:])):
            if i < 8:
                tm[i+1]['K1'].append(testing.loc[testing.loc[testing[1]=="ТМ1"].index[0]+1,:][i+1])
                tm[i+1]['K2'].append(testing.loc[testing.loc[testing[1]=="ТМ1"].index[0]+3,:][i+1])
    for i in tm.keys():
        for j in tm[i].keys():
            for x in range(len(tm[i][j])):
                tm[i][j][x] =  tm[i][j][x].hour * 3600 + tm[i][j][x].minute * 60  + tm[i][j][x].second  
            tm[i][j] = str(pd.to_datetime(sum(tm[i][j]), unit = 's')).split(' ')[1]
    for x in sheets:
        x = x.split('(')[1].split(')')[0]
        tm_counts[x].append(1)
    for y in tm_counts.keys():
        tm_counts[y] = sum(tm_counts[y])
    proppelant['Сумма расходов'] = sum(proppelant['Сумма расходов'])
    proppelant['Разность расходов'] = (pd.read_excel(files[0], sheet_name = sheets[0]).iloc[1, -4] - pd.read_excel(files[0], sheet_name = sheets[-1]).iloc[1, -3]) * 1000

In [18]:
def saveToExcel():
    FOS = []
    GN = []
    GN_values = []
    FOS_values = []
    
    Kv = []
    Sv = []
    Sv_values = []
    Kv_values = []
    for x in count1.keys():
        if 'ГН' in x:
            GN.append(x)
            GN_values.append(count1[x])
        else:
            FOS.append(x)
            FOS_values.append(count1[x])
            
    for x in count.keys():
        if 'Вкл' in x:
            Kv.append(x)
            Kv_values.append(count[x])
        else:
            Sv.append(x)
            Sv_values.append(count[x])
    directory = filedialog.askdirectory()

    
    with pd.ExcelWriter(f'{directory}/Наработки.xlsx') as writer:
        if len(count) > 0:
            pd.DataFrame(columns = ['Вкл ТМ'], index = Kv, data = Kv_values).to_excel(writer, sheet_name='KvTM')
            pd.DataFrame(columns = ['Время ТМ и БПК'], index = Sv, data = Sv_values).to_excel(writer, sheet_name='SovTM')
        if len(count1) > 0:
            pd.DataFrame(columns = ['ГН'], index = GN, data = GN_values).to_excel(writer, sheet_name='ГН')
            pd.DataFrame(columns = ['ФОС'], index = FOS, data = FOS_values).to_excel(writer, sheet_name='FOS')
        if len(tm) > 0:
            pd.DataFrame(tm).to_excel(writer, sheet_name='ТМ по excel')
            #tm_counts1 = 
            pd.DataFrame(tm_counts, index = [1]).to_excel(writer, sheet_name='Количество импульсов') 
            pd.DataFrame(proppelant, index = [1]).to_excel(writer, sheet_name='Топливо') 

In [ ]:
class MainWindow(QWidget):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Наработки K-2")
        self.setFixedSize(250, 110)

        # Кнопка 1
        btn1 = QPushButton("ГН, KVTM, ФОС, SOVTM", self)
        btn1.setGeometry(20, 10, 210, 25)
        btn1.clicked.connect(GnFosSovKv)

        # Кнопка 2
        btn2 = QPushButton("Подсчет ТМ по excel", self)
        btn2.setGeometry(20, 40, 210, 25)
        btn2.clicked.connect(excelTM)

        # Кнопка 3
        btn3 = QPushButton("Сохранение в excel", self)
        btn3.setGeometry(20, 70, 210, 25)
        btn3.clicked.connect(saveToExcel)

if __name__ == "__main__":
    app = QApplication(sys.argv)
    w = MainWindow()
    w.show()
    app.exec()